In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.mathtext import _mathtext as mathtext
from sklearn.metrics import mean_squared_error

mathtext.FontConstantsBase.sup1 = 0.45

plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'


In [ ]:
datasets = [
    {
        'path': '../../data/supplementary_figures/figure_level/figS03_hutchison_validation.csv',
        'ylabel': 'Hutchison et al.',
        'unit': 'DM ha$^{-1}$'
    },
    {
        'path': '../../data/supplementary_figures/figure_level/figS03_hu_validation.csv',
        'ylabel': 'Hu et al.',
        'unit': 'DM ha$^{-1}$'
    },
    {
        'path': '../../data/supplementary_figures/figure_level/figS03_simard_validation.csv',
        'ylabel': 'Simard et al.',
        'unit': 'DM ha$^{-1}$'
    }
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.subplots_adjust(wspace=0.25)

for idx, config in enumerate(datasets):
    ax = axes[idx]
    results = pd.read_csv(config['path'])
    yobs = results['Observed_AGB']
    ymod = results['Product_AGB']
    valid = ~(np.isnan(yobs) | np.isnan(ymod))
    yobs = yobs[valid]
    ymod = ymod[valid]

    R2 = np.corrcoef(yobs, ymod)[0, 1] ** 2
    RMSE = np.sqrt(mean_squared_error(yobs, ymod))
    b = np.polyfit(yobs, ymod, 1)

    ax.scatter(yobs, ymod, edgecolor='gray', facecolor='none', s=30)
    ax.plot([-50, 850], [-50, 850], '--', color='gray', lw=1)
    ax.plot([-50, 850], np.polyval(b, [-50, 850]), 'r-', lw=1, alpha=0.5)
    ax.set(
        xlim=(-50, 850),
        ylim=(-50, 850),
        xticks=[0, 200, 400, 600, 800],
        yticks=[0, 200, 400, 600, 800],
        xlabel='Observed AGB (Mg DM ha$^{-1}$)',
        ylabel=f'AGB from {config["ylabel"]} (Mg {config["unit"]})'
    )
    ax.set_title(f'{chr(97 + idx)}', loc='left', fontweight='bold')
    ax.text(
        50, 600,
        f'y = {b[0]:.2f}x + {b[1]:.2f}\n'
        f'R$^{{2}}$ = {R2:.2f}\n'
        f'RMSE = {RMSE:.0f} Mg {config["unit"]}'
    )

plt.savefig('../../figures/supplementary/figS03_validation_other_products.png', dpi=400, bbox_inches='tight')
plt.show()
